## Hugging face pipeline testing

simple notebook for testing various hugging face pipelines across different nlp and vision tasks.

In [ ]:
from typing import List, Dict, Any
from transformers import pipeline
from IPython.display import Image, display
import torch
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = 0 if torch.cuda.is_available() else -1

### Text summarization
condenses long text into shorter summaries using pre-trained transformer models.

In [ ]:
SAMPLE_TEXT = """
The Hub is home to over 5,000 datasets in more than 100 languages that can be used for a broad range of tasks across NLP, Computer Vision, and Audio. The Hub makes it simple to find, download, and upload datasets.
Datasets are accompanied by extensive documentation in the form of Dataset Cards and Dataset Preview to let you explore the data directly in your browser.
While many datasets are public, organizations and individuals can create private datasets to comply with licensing or privacy issues. You can learn more about Datasets here on Hugging Face Hub documentation.

The 🤗 datasets library allows you to programmatically interact with the datasets, so you can easily use datasets from the Hub in your projects.
With a single line of code, you can access the datasets; even if they are so large they don't fit in your computer, you can use streaming to efficiently access the data.
"""

def summarize_text(text: str, max_len: int = 130, min_len: int = 30) -> str:
    """Generate summary using distilbart model."""
    summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6", device=DEVICE)
    results: Any = summarizer(text, max_length=max_len, min_length=min_len, do_sample=False)
    if not results:
        return ""
    summary_text = results[0].get("summary_text", "")
    return str(summary_text)

summary = summarize_text(SAMPLE_TEXT)
print(f"Summary:\n{summary}")

### Text generation
generates text continuation from a given prompt using language models.

In [ ]:
PROMPT = "Scientists from University of California, Berkeley has announced that"

def generate_text(prompt: str, max_len: int = 100, num_seq: int = 1, num_beams: int = 3, temp: float = 0.8) -> List[str]:
    """Generate text continuations with controlled randomness."""
    generator = pipeline("text-generation", model="gpt2", device=DEVICE)
    results = generator(
        prompt, 
        max_length=max_len,
        truncation=True,
        num_return_sequences=num_seq,
        temperature=temp,
        do_sample=True,
        num_beams=num_beams,
        pad_token_id=generator.tokenizer.eos_token_id
    )
    return [r['generated_text'] for r in results]

generated = generate_text(PROMPT)
print(f"Generated:\n{generated[0]}")

### Zero-shot classification
classifies text into categories without specific training, using semantic understanding.

In [ ]:
SAMPLE_TEXT = """
Transformers support framework interoperability between PyTorch, TensorFlow, and JAX. 
This provides the flexibility to use a different framework at each stage of a model's life; 
train a model in three lines of code in one framework, and load it for inference in another.
"""

def classify_text(text: str, labels: List[str], template: str = "This text is about {}.") -> Dict:
    """Classify text into categories using zero-shot learning."""
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=DEVICE)
    return classifier(text, candidate_labels=labels, hypothesis_template=template)

labels = ["technology", "biology", "space", "transport"]
results = classify_text(SAMPLE_TEXT, labels)


print("Classification results:")
for label, score in zip(results['labels'], results['scores']):
    print(f"  {label}: {score:.4f}")

### Image captioning
generates descriptive text captions for images using vision-language models.

In [ ]:
IMAGE_URL = 'https://huggingface.co/datasets/Narsil/image_dummy/raw/main/parrots.png'

def generate_caption(image_url: str, model: str = "ydshieh/vit-gpt2-coco-en") -> str:
    """Generate descriptive caption for image using vision-language model."""
    captioner = pipeline("image-to-text", model=model, device=DEVICE)
    result = captioner(image_url)
    return result[0]['generated_text']

display(Image(url=IMAGE_URL))
caption = generate_caption(IMAGE_URL)
print(f"Caption: {caption}")